# 🚀 NeoKarir — Otak 1: Intent Classifier dengan IndoBERT

**Tujuan Notebook ini:**
Fine-tune model `indobenchmark/indobert-base-p2` dari HuggingFace untuk mengklasifikasikan intent pengguna chatbot NeoKarir ke dalam **5 kategori**:

| Intent | Deskripsi |
|--------|-----------|
| `bantuan_fitur_aplikasi` | Pertanyaan tentang cara pakai fitur di aplikasi |
| `tanya_syarat_loker` | Pertanyaan skill/syarat untuk posisi pekerjaan tertentu |
| `tanya_roadmap` | Pertanyaan urutan/jalur belajar |
| `salam_sapaan` | Basa-basi, salam, terima kasih |
| `out_of_context` | Pertanyaan di luar topik karir |

**Stack:** HuggingFace `transformers` + `datasets` + `evaluate` + `scikit-learn` + PyTorch

---
> ⚠️ **Pastikan runtime Google Colab kamu menggunakan GPU!**  
> Klik: `Runtime` → `Change runtime type` → pilih `T4 GPU`

## Cell 1 — Setup & Install Library

Kita install semua library yang dibutuhkan. Penjelasan singkat tiap library:
- **`transformers`**: Library utama dari HuggingFace untuk load/fine-tune model BERT, GPT, dsb.
- **`datasets`**: Untuk membuat dan memanipulasi dataset dengan format yang kompatibel dengan `Trainer`.
- **`evaluate`**: Library HuggingFace untuk menghitung metrik evaluasi model.
- **`scikit-learn`**: Library ML klasik Python, kita pakai untuk metrik Precision, Recall, F1.
- **`accelerate`**: Dependency dari `transformers` untuk training yang lebih efisien.

In [1]:
# ============================================================
# CELL 1: Setup & Install
# ============================================================

# Install semua library yang diperlukan
# Flag -q (quiet) agar output instalasi tidak terlalu panjang
!pip install -q transformers datasets evaluate scikit-learn accelerate

# Verifikasi versi PyTorch dan ketersediaan GPU
import torch

print("="*50)
print(f"✅ PyTorch version : {torch.__version__}")
print(f"✅ CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU tidak terdeteksi. Training akan lambat! Cek runtime Colab kamu.")
print("="*50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
✅ PyTorch version : 2.10.0+cu128
✅ CUDA available  : True
✅ GPU             : Tesla T4


## Cell 2 — Buat Dataset Training & Validasi

Karena ini proyek capstone, kita buat dataset secara manual terlebih dahulu (hardcoded).
Dalam produksi nyata, kamu bisa memuat dari file CSV dengan `pd.read_csv()`.

**Tips:** Untuk model yang bagus, minimal butuh **50–100 sampel per intent**. Di sini kita buat dataset kecil sebagai template — **kamu harus menambah data lebih banyak!**

**Split data:** 80% training, 20% validasi — standar industri untuk dataset kecil.

In [2]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("/kaggle/input/datasets/laventiliz/chatbot-intent-neokarir/intent_dataset_final_v2.csv")
print(f"📊 Total data: {len(df)} sampel")
print("\n📈 Distribusi per intent:")
print(df["intent"].value_counts())
df.head()

📊 Total data: 1819 sampel

📈 Distribusi per intent:
intent
analisis_skill_gap        335
bantuan_fitur_aplikasi    332
cari_lowongan             301
tanya_roadmap_karir       275
tanya_tips_rekrutmen      243
out_of_context            225
salam_sapaan              108
Name: count, dtype: int64


,chat_text,intent
0,ada lowongan ui/ux designer yang full wfh ngga,cari_lowongan
1,makasih,salam_sapaan
2,bsa tolong translate dokumen ini ke bahasa ing...,out_of_context
3,gimana cara pakai job match score,bantuan_fitur_aplikasi
4,"cekin profil saya dong, udah cocok belum lamar...",analisis_skill_gap


In [4]:
# ------------------------------------------------------------
# Split data menjadi train (80%) dan validation (20%)
# stratify=df["label"] → pastikan proporsi setiap kelas
# terjaga di kedua split (penting untuk dataset kecil!)
# random_state=42 → agar hasil reproducible (bisa diulang)
# ------------------------------------------------------------
df_train, df_val = train_test_split(
    df,
    test_size=0.2,
    stratify=df["intent"],
    random_state=42
)

print(f"✅ Data training  : {len(df_train)} sampel")
print(f"✅ Data validasi  : {len(df_val)} sampel")

# Reset index penting agar HuggingFace Dataset tidak error
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

# Konversi DataFrame pandas ke HuggingFace Dataset
# Format DatasetDict adalah standar yang diharapkan oleh Trainer HuggingFace
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "validation": Dataset.from_pandas(df_val)
})

print("\n📦 Dataset HuggingFace:")
print(raw_datasets)

✅ Data training  : 1455 sampel
✅ Data validasi  : 364 sampel

📦 Dataset HuggingFace:
DatasetDict({
    train: Dataset({
        features: ['chat_text', 'intent'],
        num_rows: 1455
    })
    validation: Dataset({
        features: ['chat_text', 'intent'],
        num_rows: 364
    })
})


## Cell 3 — Tokenisasi & Label Encoding

Ini adalah cell **paling kritis** dalam pipeline.

**Kenapa perlu Label Encoding?**
Model deep learning hanya bisa memproses angka. Label seperti `"tanya_roadmap"` harus diubah menjadi integer seperti `2`. Kita buat mapping dua arah:
- `label2id`: string → integer (untuk training)
- `id2label`: integer → string (untuk inference/prediksi)

**Kenapa pakai `AutoTokenizer`?**
Tokenizer IndoBERT mengubah kalimat menjadi token-token yang sesuai dengan vocabulary model tersebut. Harus menggunakan tokenizer yang *sama* dengan model yang akan dipakai.

**`truncation=True` & `padding=True`:**
- `truncation`: potong kalimat yang terlalu panjang (> `max_length` token)
- `padding`: tambah padding agar semua kalimat dalam satu batch punya panjang yang sama

In [5]:
# ============================================================
# CELL 3: Tokenisasi & Label Encoding
# ============================================================

from transformers import AutoTokenizer

# ------------------------------------------------------------
# KONFIGURASI MODEL
# Gunakan indobert-base-p2 yang dilatih khusus untuk Bahasa Indonesia
# ------------------------------------------------------------
MODEL_CHECKPOINT = "indobenchmark/indobert-base-p2"

# Load tokenizer sesuai checkpoint
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

print(f"✅ Tokenizer dimuat dari: {MODEL_CHECKPOINT}")

# ------------------------------------------------------------
# LABEL ENCODING
# Ambil semua label unik dari dataset dan urutkan agar konsisten
# (urutan alfabetis = deterministik, tidak bergantung pada urutan data)
# ------------------------------------------------------------
labels_unik = sorted(df["intent"].unique().tolist())
print(f"\n🏷️  Label yang ditemukan ({len(labels_unik)} intent): {labels_unik}")

# Buat mapping: string label → integer ID
label2id = {label: idx for idx, label in enumerate(labels_unik)}

# Buat mapping kebalikannya: integer ID → string label
# Ini dibutuhkan saat model melakukan prediksi — output model berupa angka,
# kita perlu terjemahkan balik ke nama intent yang readable.
id2label = {idx: label for label, idx in label2id.items()}

print("\n🔢 Mapping label2id:")
for k, v in label2id.items():
    print(f"   '{k}' → {v}")

# Jumlah kelas (dibutuhkan saat inisialisasi model)
NUM_LABELS = len(labels_unik)
print(f"\n✅ Jumlah kelas (NUM_LABELS): {NUM_LABELS}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Tokenizer dimuat dari: indobenchmark/indobert-base-p2

🏷️  Label yang ditemukan (7 intent): ['analisis_skill_gap', 'bantuan_fitur_aplikasi', 'cari_lowongan', 'out_of_context', 'salam_sapaan', 'tanya_roadmap_karir', 'tanya_tips_rekrutmen']

🔢 Mapping label2id:
   'analisis_skill_gap' → 0
   'bantuan_fitur_aplikasi' → 1
   'cari_lowongan' → 2
   'out_of_context' → 3
   'salam_sapaan' → 4
   'tanya_roadmap_karir' → 5
   'tanya_tips_rekrutmen' → 6

✅ Jumlah kelas (NUM_LABELS): 7


In [6]:
# ------------------------------------------------------------
# FUNGSI PREPROCESSING
# Fungsi ini akan dijalankan pada setiap baris dataset
# menggunakan metode .map() dari HuggingFace Dataset
# ------------------------------------------------------------
def preprocess_function(examples):
    """
    Melakukan dua hal sekaligus:
    1. Tokenisasi teks
    2. Konversi string label → integer

    Parameter:
    - truncation=True : potong kalimat yang melebihi max_length
    - padding='max_length': pad kalimat pendek sampai max_length
      (alternatif: padding=True untuk dynamic padding per batch)
    - max_length=128  : 128 token sudah cukup untuk kalimat pendek
                         seperti query chatbot. Menaikkan ke 512
                         butuh lebih banyak VRAM GPU.
    """
    # Langkah 1: Tokenisasi teks
    tokenized = tokenizer(
        examples["chat_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # Langkah 2: Konversi string label → integer
    # KRITIS: Jika kita tidak melakukan ini, Trainer akan error
    # karena model mengharapkan kolom 'labels' berupa integer, bukan string.
    # Nama kolom HARUS 'labels' (bukan 'label') agar Trainer otomatis mengenalinya.
    tokenized["labels"] = [label2id[lbl] for lbl in examples["intent"]]

    return tokenized

# Terapkan fungsi preprocessing ke seluruh dataset
# batched=True → proses data dalam batch (jauh lebih cepat)
# remove_columns=["text", "label"] → hapus kolom asli yang sudah tidak diperlukan
#   (Trainer hanya butuh: input_ids, attention_mask, token_type_ids, labels)
print("⏳ Memproses dataset...")

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["chat_text", "intent"]  # Hapus kolom teks asli
)

print("\n✅ Dataset setelah tokenisasi:")
print(tokenized_datasets)

print("\n🔍 Contoh satu sampel dari training set:")
sample = tokenized_datasets["train"][0]
print(f"   input_ids (5 pertama) : {sample['input_ids'][:5]}")
print(f"   attention_mask (5 pertama): {sample['attention_mask'][:5]}")
print(f"   labels                : {sample['labels']} → '{id2label[sample['labels']]}'")

⏳ Memproses dataset...


Map:   0%|          | 0/1455 [00:00<?, ? examples/s]

Map:   0%|          | 0/364 [00:00<?, ? examples/s]


✅ Dataset setelah tokenisasi:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1455
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 364
    })
})

🔍 Contoh satu sampel dari training set:
   input_ids (5 pertama) : [2, 1, 2526, 24137, 744]
   attention_mask (5 pertama): [1, 1, 1, 1, 1]
   labels                : 3 → 'out_of_context'


## Cell 4 — Data Collator & Metrik Evaluasi

**Data Collator** bertugas mengumpulkan sampel-sampel individual menjadi satu batch selama training. `DataCollatorWithPadding` akan otomatis melakukan padding dinamis (tiap batch di-pad hanya sampai panjang maksimum dalam batch tersebut, bukan panjang maksimum global). Ini lebih efisien dari segi memori.

**Metrik Evaluasi — Kenapa Macro F1?**
- **Macro Average**: Hitung F1 per kelas, lalu rata-rata *tanpa* mempertimbangkan ukuran kelas. Ini adil untuk kelas yang seimbang.
- **Micro Average**: Hitung F1 secara global. Cenderung bias ke kelas mayoritas.
- **Weighted Average**: Seperti macro, tapi dikali proporsi tiap kelas. Dipengaruhi oleh kelas besar.

Untuk chatbot intent classification dengan kelas yang seimbang, **Macro F1** adalah pilihan yang tepat karena kita peduli pada performa di *setiap* intent secara merata.

In [7]:
# ============================================================
# CELL 4: Data Collator & Fungsi Metrik
# ============================================================

import numpy as np
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ------------------------------------------------------------
# DATA COLLATOR
# Mengatur bagaimana sampel-sampel digabung menjadi batch.
# DataCollatorWithPadding lebih efisien dari padding statis
# karena hanya pad sampai panjang terpanjang dalam satu batch.
# ------------------------------------------------------------
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Data Collator siap.")

# ------------------------------------------------------------
# FUNGSI COMPUTE METRICS
# Dipanggil secara otomatis oleh Trainer HuggingFace di akhir setiap epoch.
# Input: EvalPrediction object yang berisi logits & label ID asli.
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    """
    Menghitung metrik evaluasi dari prediksi model.

    Alur:
    1. logits → argmax → predictions (indeks kelas dengan skor tertinggi)
    2. Bandingkan predictions vs labels asli menggunakan scikit-learn
    3. Return dict berisi Accuracy, Precision, Recall, F1
    """
    logits, labels = eval_pred

    # logits adalah skor mentah (belum softmax) dari model.
    # np.argmax → ambil indeks kelas dengan skor tertinggi = prediksi
    predictions = np.argmax(logits, axis=-1)

    # Hitung Precision, Recall, F1
    # average='macro' → rata-rata TIDAK tertimbang antar kelas
    #   Cocok ketika kita ingin performa merata di semua intent,
    #   bukan hanya intent yang sering muncul.
    # zero_division=0 → jika suatu kelas tidak ada prediksi sama sekali,
    #   anggap nilai metriknya 0 (hindari error ZeroDivisionError)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    # Hitung akurasi standar (% prediksi benar dari total)
    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy"  : round(accuracy, 4),
        "f1_macro"  : round(f1, 4),
        "precision_macro": round(precision, 4),
        "recall_macro"   : round(recall, 4),
    }

print("✅ Fungsi compute_metrics siap.")

✅ Data Collator siap.
✅ Fungsi compute_metrics siap.


## Cell 5 — Inisialisasi Model

Kita load `indobert-base-p2` sebagai base model, lalu tambahkan **classification head** di atasnya menggunakan `AutoModelForSequenceClassification`.

**Apa itu fine-tuning?**  
IndoBERT sudah dilatih pada corpus bahasa Indonesia yang sangat besar (pre-training). Kita tidak melatih dari nol — kita hanya "menyesuaikan" (fine-tune) bobot modelnya pada task spesifik kita (intent classification). Ini jauh lebih efisien dan butuh data lebih sedikit.

**Kenapa `ignore_mismatched_sizes=True`?**  
Layer classifier (head) di model pre-trained memiliki ukuran yang berbeda dengan yang kita butuhkan (`num_labels=5`). Parameter ini memberitahu HuggingFace untuk mengabaikan mismatch tersebut dan melakukan reinisialisasi layer tersebut — ini normal dan diharapkan!

In [8]:
# ============================================================
# CELL 5: Inisialisasi Model
# ============================================================

from transformers import AutoModelForSequenceClassification

# Load model IndoBERT dengan classification head
# num_labels   → jumlah output neuron (= jumlah intent kita = 5)
# id2label     → mapping untuk digunakan saat inference
# label2id     → mapping untuk digunakan saat training
# ignore_mismatched_sizes=True → WAJIB untuk fine-tuning!
#   Model asli mungkin punya head dengan ukuran berbeda.
#   Parameter ini memberitahu HuggingFace untuk inisialisasi
#   ulang layer classifier sesuai num_labels kita.
print("⏳ Memuat model IndoBERT... (mungkin perlu beberapa menit pertama kali)")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  # Penting untuk fine-tuning!
)

# Pindahkan model ke GPU jika tersedia
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"\n✅ Model berhasil dimuat!")
print(f"   Device : {device}")
print(f"   Labels : {model.config.id2label}")

# Hitung total parameter yang bisa ditraining
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Total parameter    : {total_params:,}")
print(f"📊 Parameter trainable: {trainable_params:,}")

⏳ Memuat model IndoBERT... (mungkin perlu beberapa menit pertama kali)


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]


✅ Model berhasil dimuat!
   Device : cuda
   Labels : {0: 'analisis_skill_gap', 1: 'bantuan_fitur_aplikasi', 2: 'cari_lowongan', 3: 'out_of_context', 4: 'salam_sapaan', 5: 'tanya_roadmap_karir', 6: 'tanya_tips_rekrutmen'}

📊 Total parameter    : 124,446,727
📊 Parameter trainable: 124,446,727


## Cell 6 — Training Setup & Jalankan Training

**Penjelasan Hyperparameter Penting:**

| Parameter | Nilai | Alasan |
|-----------|-------|--------|
| `num_train_epochs` | 10 | Dataset kecil butuh lebih banyak epoch. Pantau jika overfitting. |
| `learning_rate` | 2e-5 | Standard untuk fine-tuning BERT. Terlalu besar → merusak pre-trained weights. |
| `per_device_train_batch_size` | 16 | Ukuran batch per GPU. Sesuaikan jika OOM (Out of Memory). |
| `weight_decay` | 0.01 | Regularisasi L2 untuk mencegah overfitting. |
| `eval_strategy` | `"epoch"` | Evaluasi di akhir setiap epoch. |
| `load_best_model_at_end` | `True` | Otomatis load checkpoint terbaik setelah training selesai. |
| `metric_for_best_model` | `"f1_macro"` | Simpan model berdasarkan F1 macro terbaik, bukan loss. |

In [9]:
# ============================================================
# CELL 6: Training Setup & Eksekusi
# ============================================================

from transformers import TrainingArguments, Trainer

# ------------------------------------------------------------
# TRAINING ARGUMENTS
# Konfigurasi lengkap proses training
# ------------------------------------------------------------
training_args = TrainingArguments(
    # Direktori untuk menyimpan checkpoint selama training
    output_dir="./neokarir_intent_checkpoints",

    # === Hyperparameter Utama ===
    num_train_epochs=5,          # Jumlah epoch. Dataset kecil → butuh lebih banyak epoch.
    learning_rate=2e-5,           # Learning rate standar untuk fine-tuning BERT.
                                  # Range rekomendasi: 1e-5 s/d 5e-5.

    # === Konfigurasi Batch ===
    per_device_train_batch_size=16,  # Batch size training per GPU.
    per_device_eval_batch_size=16,   # Batch size evaluasi per GPU.
                                     # Jika GPU OOM, turunkan ke 8.

    # === Regularisasi (Mencegah Overfitting) ===
    weight_decay=0.01,            # L2 regularization. Nilai kecil sudah cukup untuk BERT.

    # === Evaluasi & Checkpoint ===
    eval_strategy="epoch",        # Evaluasi di akhir setiap epoch
    save_strategy="epoch",        # Simpan checkpoint di akhir setiap epoch
    load_best_model_at_end=True,  # PENTING: Setelah training, load model TERBAIK,
                                  # bukan model dari epoch terakhir.
    metric_for_best_model="f1_macro",   # Kriteria 'terbaik' berdasarkan F1 macro
    greater_is_better=True,             # F1 yang lebih tinggi = lebih baik

    # === Logging ===
    logging_dir="./neokarir_logs",
    logging_steps=10,             # Log setiap 10 langkah training

    # === Lain-lain ===
    warmup_ratio=0.1,             # 10% dari total steps untuk warmup learning rate
                                  # Mencegah perubahan bobot yang terlalu drastis di awal.
    save_total_limit=2,           # Hanya simpan 2 checkpoint terbaik (hemat disk)
    fp16=torch.cuda.is_available(), # Gunakan FP16 (half precision) jika ada GPU
                                    # → Training 2x lebih cepat, setengah memori GPU!
    report_to="none",             # Nonaktifkan WandB/MLflow untuk menyederhanakan output
)

# ------------------------------------------------------------
# INISIALISASI TRAINER
# HuggingFace Trainer menyederhanakan training loop yang kompleks
# menjadi satu objek yang mudah digunakan.
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer siap! Memulai training...")
print("="*60)

# ------------------------------------------------------------
# MULAI TRAINING!
# ------------------------------------------------------------
train_result = trainer.train()

print("\n" + "="*60)
print("🎉 Training selesai!")
print(f"   Total waktu training : {train_result.metrics['train_runtime']:.2f} detik")
print(f"   Training loss akhir  : {train_result.metrics['train_loss']:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Trainer siap! Memulai training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,1.183193,0.383908,0.980800,0.980300,0.985200,0.977800
2,0.029722,0.017890,1.000000,1.000000,1.000000,1.000000
3,0.012160,0.008391,1.000000,1.000000,1.000000,1.000000
4,0.008545,0.006542,1.000000,1.000000,1.000000,1.000000
5,0.007774,0.006100,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


🎉 Training selesai!
   Total waktu training : 120.11 detik
   Training loss akhir  : 0.5002


In [10]:
# ------------------------------------------------------------
# EVALUASI FINAL pada validation set
# Ini menggunakan model terbaik (karena load_best_model_at_end=True)
# ------------------------------------------------------------
print("\n📊 Evaluasi final pada Validation Set:")
print("-"*40)

eval_results = trainer.evaluate()

for metric, value in eval_results.items():
    if not metric.startswith("eval_runtime") and not metric.startswith("eval_samples"):
        print(f"   {metric:30s}: {value:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



📊 Evaluasi final pada Validation Set:
----------------------------------------


   eval_loss                     : 0.0178
   eval_accuracy                 : 1.0000
   eval_f1_macro                 : 1.0000
   eval_precision_macro          : 1.0000
   eval_recall_macro             : 1.0000
   eval_steps_per_second         : 6.2070
   epoch                         : 5.0000


## Cell 7 — Inference Pipeline (Uji Model)

Setelah training, kita uji model menggunakan HuggingFace `pipeline()`. Ini adalah cara paling mudah untuk melakukan inferensi dengan model yang sudah dilatih.

Pipeline akan otomatis:
1. Tokenisasi input
2. Forward pass melalui model
3. Softmax untuk mendapatkan probabilitas
4. Return label dan score

Kita akan uji dengan **3 kalimat tricky** untuk melihat kemampuan generalisasi model.

In [11]:
# ============================================================
# CELL 7: Inference Pipeline — Uji Model
# ============================================================

from transformers import pipeline

# Buat pipeline text-classification menggunakan model dan tokenizer
# yang baru saja kita fine-tune.
# device=0 → gunakan GPU pertama jika tersedia, -1 untuk CPU
intent_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

print("✅ Pipeline siap!\n")
print("=" * 65)

# ------------------------------------------------------------
# 3 KALIMAT UJI YANG TRICKY
# Dipilih untuk menguji kemampuan generalisasi model:
# 1. Ambigu antara fitur aplikasi dan syarat loker
# 2. Kalimat roadmap yang tidak eksplisit
# 3. Out of context yang bisa dikira karir
# ------------------------------------------------------------
kalimat_uji = [
  {
    "teks": "IPK gw cuma 2.5 lulusan soshum, jujur aja masih ada harapan ga sih gw tembus seleksi berkas di startup tech?",
    "ekspektasi": "tanya_tips_rekrutmen",
    "alasan_susah": "Penuh dengan kecemasan emosional tanpa menyebut satupun nama role IT (seperti frontend/backend) atau skill teknis. Inti pertanyaannya adalah mencari tips atau insight tentang proses 'seleksi berkas'."
  },
  {
    "teks": "Min, format portofolio yang bener tuh mending disimpen pake link GDrive trus ditaruh di bio, atau wajib di-upload file-nya ke sistem lu?",
    "ekspektasi": "bantuan_fitur_aplikasi",
    "alasan_susah": "Ada kata 'portofolio', 'format yang bener'. Model rawan lari ke `tanya_tips_rekrutmen`, padahal inti pertanyaannya adalah cara kerja UI/sistem aplikasi NeoKarir."
  },
  {
    "teks": "Gw udah tamat nih namatin semua course React sama Node JS, sekarang ada tempat yang mau nerima gw ga ya buat praktek kerja langsung?",
    "ekspektasi": "cari_lowongan",
    "alasan_susah": "Kata 'tamat course', 'React', 'Node JS' sangat identik dengan `tanya_roadmap_karir`. Tapi frasa 'tempat yang mau nerima gw' adalah bahasa gaul dari 'cari loker'."
  },
  {
    "teks": "Skill gw nanggung banget di UI/UX, menurut lu gw mending fokus perdalam Figma aja atau langsung lompat nyoba koding front-end biar cepet dapet kerja?",
    "ekspektasi": "tanya_roadmap_karir",
    "alasan_susah": "Ada kata 'Skill gw nanggung' (jebakan `analisis_skill_gap`) dan 'dapet kerja' (jebakan `cari_lowongan`). Padahal user sedang meminta saran arah/jalur belajar."
  },
  {
    "teks": "Bisa tolong bedah CV gw ga min? Gw pengen tau kenapa tiap daftar posisi Data Engineer di startup ijo pasti gugur mulu di tahap screening awal.",
    "ekspektasi": "analisis_skill_gap",
    "alasan_susah": "Kata 'bedah CV' dan 'gugur di tahap screening' sangat identik dengan `tanya_tips_rekrutmen`. Tapi karena dia menyebut role spesifik 'Data Engineer' dan minta dibedah (dianalisis), ini lebih tepat ditarik ke `analisis_skill_gap`."
  },
  {
    "teks": "Bro gw butuh loker yang kerjanya cuma rebahan tapi gajinya 50 juta sebulan, lu ada kenalan orang dalem ga?",
    "ekspektasi": "out_of_context",
    "alasan_susah": "Sangat rawan masuk ke `cari_lowongan` karena ada kata 'loker' dan 'gaji'. Ini menguji apakah model bisa mendeteksi absurditas/hard negatives."
  },
  {
    "teks": "Tolong dong ajarin gw step-by-step cara bikin koneksi database di Firebase buat tugas akhir kampus, beneran mentok ini gw.",
    "ekspektasi": "out_of_context",
    "alasan_susah": "Ada kata 'step-by-step' (jebakan `tanya_roadmap_karir`) dan istilah teknis 'database Firebase'. Ini menguji *domain boundary* (bahwa bot tidak membantu tugas coding)."
  },
  {
    "teks": "Pernah ga sih lu ngerasa usernya tuh jutek banget pas lagi technical test? Kalo digituin mending kita tetep senyum kalem atau jawab seadanya aja?",
    "ekspektasi": "tanya_tips_rekrutmen",
    "alasan_susah": "Kalimatnya sangat berbentuk curhatan/drama (rawan `out_of_context`), tapi ini sebenarnya adalah pertanyaan valid tentang *attitude* saat *interview/technical test*."
  },
  {
    "teks": "Tiba-tiba halaman utama gw nge-blank putih pas nyoba sinkronin akun LinkedIn ke profil, ini harus clear cache dulu apa gimana?",
    "ekspektasi": "bantuan_fitur_aplikasi",
    "alasan_susah": "Ada kata 'akun LinkedIn' dan 'profil' (jebakan `tanya_tips_rekrutmen` atau `analisis_skill_gap`). Padahal user bertanya tentang bug/error sistem."
  }
]

for i, item in enumerate(kalimat_uji, 1):
    hasil = intent_classifier(item["teks"])[0]
    label_prediksi = hasil["label"]
    confidence     = hasil["score"]

    print(f"\n📝 Uji #{i}")
    print(f"   Input      : '{item['teks']}'")
    print(f"   Prediksi   : {label_prediksi}")
    print(f"   Confidence : {confidence:.2%}")
    print(f"   Ekspektasi : {item['ekspektasi']}")
    print(f"   alasan     : {item['alasan_susah']}")
    print("-" * 65)

✅ Pipeline siap!


📝 Uji #1
   Input      : 'IPK gw cuma 2.5 lulusan soshum, jujur aja masih ada harapan ga sih gw tembus seleksi berkas di startup tech?'
   Prediksi   : analisis_skill_gap
   Confidence : 96.87%
   Ekspektasi : tanya_tips_rekrutmen
   alasan     : Penuh dengan kecemasan emosional tanpa menyebut satupun nama role IT (seperti frontend/backend) atau skill teknis. Inti pertanyaannya adalah mencari tips atau insight tentang proses 'seleksi berkas'.
-----------------------------------------------------------------

📝 Uji #2
   Input      : 'Min, format portofolio yang bener tuh mending disimpen pake link GDrive trus ditaruh di bio, atau wajib di-upload file-nya ke sistem lu?'
   Prediksi   : tanya_tips_rekrutmen
   Confidence : 95.03%
   Ekspektasi : bantuan_fitur_aplikasi
   alasan     : Ada kata 'portofolio', 'format yang bener'. Model rawan lari ke `tanya_tips_rekrutmen`, padahal inti pertanyaannya adalah cara kerja UI/sistem aplikasi NeoKarir.
--------------------------

In [12]:
# ------------------------------------------------------------
# BONUS: Tampilkan semua probabilitas per intent
# return_all_scores=True menampilkan skor untuk SEMUA kelas,
# berguna untuk debugging dan memahami kepercayaan model.
# ------------------------------------------------------------
print("\n🔬 Analisis mendalam — probabilitas semua intent untuk kalimat uji pertama:")
print("-" * 65)

kalimat_debug = "Bro gw butuh loker yang kerjanya cuma rebahan tapi gajinya 50 juta sebulan, lu ada kenalan orang dalem ga?"

# top_k=None → return semua label
semua_hasil = intent_classifier(kalimat_debug, top_k=None)

# Urutkan dari probabilitas tertinggi ke terendah
semua_hasil_sorted = sorted(semua_hasil, key=lambda x: x["score"], reverse=True)

print(f"   Input: '{kalimat_debug}'\n")
for hasil in semua_hasil_sorted:
    bar = "█" * int(hasil["score"] * 30)
    print(f"   {hasil['label']:30s} | {bar:30s} | {hasil['score']:.2%}")


🔬 Analisis mendalam — probabilitas semua intent untuk kalimat uji pertama:
-----------------------------------------------------------------
   Input: 'Bro gw butuh loker yang kerjanya cuma rebahan tapi gajinya 50 juta sebulan, lu ada kenalan orang dalem ga?'

   cari_lowongan                  | █████████████████████████████  | 98.57%
   analisis_skill_gap             |                                | 0.50%
   out_of_context                 |                                | 0.49%
   tanya_tips_rekrutmen           |                                | 0.17%
   salam_sapaan                   |                                | 0.11%
   bantuan_fitur_aplikasi         |                                | 0.09%
   tanya_roadmap_karir            |                                | 0.07%


## Cell 8 — Simpan & Ekspor Model

Setelah training berhasil, kita simpan model dan tokenizer ke direktori lokal. File-file ini yang nantinya akan di-deploy ke backend NeoKarir.

**File yang akan disimpan:**
- `config.json`: Konfigurasi model (arsitektur, label mapping, dll)
- `model.safetensors` / `pytorch_model.bin`: Bobot model yang sudah dilatih
- `tokenizer.json` & file terkait: Tokenizer untuk preprocessing input

**Cara load model yang sudah disimpan:**
```python
from transformers import pipeline
classifier = pipeline("text-classification", model="./neokarir_intent_model")
```

In [13]:
# ============================================================
# CELL 8: Simpan & Ekspor Model untuk Deployment
# ============================================================

import os
import json

# Tentukan direktori penyimpanan
SAVE_DIR = "./neokarir_intent_model"

print(f"💾 Menyimpan model ke: '{SAVE_DIR}'")

# Simpan model (bobot + konfigurasi)
model.save_pretrained(SAVE_DIR)
print("   ✅ Model tersimpan.")

# Simpan tokenizer (diperlukan untuk preprocessing saat inference)
tokenizer.save_pretrained(SAVE_DIR)
print("   ✅ Tokenizer tersimpan.")

# Simpan metadata tambahan yang berguna untuk deployment
metadata = {
    "project"        : "NeoKarir",
    "component"      : "Otak 1 - Intent Classifier",
    "base_model"     : MODEL_CHECKPOINT,
    "num_labels"     : NUM_LABELS,
    "label2id"       : label2id,
    "id2label"       : {str(k): v for k, v in id2label.items()},  # JSON butuh string key
    "max_length"     : 128,
    "framework"      : "pytorch",
    "task"           : "text-classification",
}

metadata_path = os.path.join(SAVE_DIR, "neokarir_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("   ✅ Metadata NeoKarir tersimpan.")

# Tampilkan struktur file yang tersimpan
print(f"\n📁 Isi direktori '{SAVE_DIR}':")
for file in sorted(os.listdir(SAVE_DIR)):
    file_path = os.path.join(SAVE_DIR, file)
    size_kb = os.path.getsize(file_path) / 1024
    print(f"   {file:40s} ({size_kb:>10.1f} KB)")

print("\n" + "="*60)
print("🚀 Model siap untuk deployment backend NeoKarir!")
print("="*60)

💾 Menyimpan model ke: './neokarir_intent_model'


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   ✅ Model tersimpan.
   ✅ Tokenizer tersimpan.
   ✅ Metadata NeoKarir tersimpan.

📁 Isi direktori './neokarir_intent_model':
   config.json                              (       1.6 KB)
   model.safetensors                        (  486143.0 KB)
   neokarir_metadata.json                   (       0.6 KB)
   tokenizer.json                           (     692.4 KB)
   tokenizer_config.json                    (       0.3 KB)

🚀 Model siap untuk deployment backend NeoKarir!


In [14]:
# ------------------------------------------------------------
# VERIFIKASI: Load ulang model dari direktori yang tersimpan
# dan pastikan inference bekerja dengan benar.
# Ini mensimulasikan apa yang dilakukan backend saat startup.
# ------------------------------------------------------------
print("🔄 Verifikasi: Memuat ulang model dari disk...")

from transformers import pipeline

# Ini adalah cara backend NeoKarir akan load model
classifier_dari_disk = pipeline(
    "text-classification",
    model=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1
)

# Tes cepat
tes_akhir = [
    "Halo, selamat pagi!",
    "Langkah belajar jadi data engineer itu gimana?",
    "Cara upload CV di NeoKarir?",
    "Skill apa yang dibutuhkan untuk jadi DevOps?",
    "Siapa yang menang Piala Dunia 2022?",
]

print("\n✅ Model loaded dari disk. Hasil tes akhir:")
print("-" * 60)
for teks in tes_akhir:
    hasil = classifier_dari_disk(teks)[0]
    print(f"   [{hasil['label']:30s} | {hasil['score']:.1%}] {teks}")

print("\n🎊 Semua selesai! NeoKarir Intent Classifier siap digunakan.")

🔄 Verifikasi: Memuat ulang model dari disk...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


✅ Model loaded dari disk. Hasil tes akhir:
------------------------------------------------------------
   [salam_sapaan                   | 98.4%] Halo, selamat pagi!
   [tanya_roadmap_karir            | 98.9%] Langkah belajar jadi data engineer itu gimana?
   [bantuan_fitur_aplikasi         | 93.8%] Cara upload CV di NeoKarir?
   [tanya_tips_rekrutmen           | 58.7%] Skill apa yang dibutuhkan untuk jadi DevOps?
   [tanya_tips_rekrutmen           | 83.2%] Siapa yang menang Piala Dunia 2022?

🎊 Semua selesai! NeoKarir Intent Classifier siap digunakan.


---
## 📋 Ringkasan & Langkah Selanjutnya

### ✅ Apa yang sudah kita bangun:
1. **Dataset** — 75 sampel berlabel (15 per intent) dengan stratified split 80/20
2. **Tokenizer** — IndoBERT tokenizer dengan max_length=128
3. **Model** — IndoBERT fine-tuned untuk 5-class intent classification
4. **Training** — Dengan evaluasi per epoch, early stopping via best model loading
5. **Inference Pipeline** — Siap diintegrasikan ke backend
6. **Saved Model** — Tersimpan di `./neokarir_intent_model/`

### 🚀 Langkah Selanjutnya untuk Production:

1. **Tambah Data** — Target minimal 100 sampel per intent (500 total). Kumpulkan dari real user queries.
2. **Evaluasi Lebih Lanjut** — Buat confusion matrix untuk lihat intent mana yang sering salah klasifikasi.
3. **Threshold Confidence** — Implementasikan fallback: jika confidence < 70%, arahkan ke `out_of_context`.
4. **Export ke ONNX** — Untuk inference yang lebih cepat di production server:
   ```python
   # pip install optimum
   from optimum.onnxruntime import ORTModelForSequenceClassification
   ```
5. **Upload ke HuggingFace Hub** — Agar mudah di-load dari mana saja:
   ```python
   trainer.push_to_hub("username/neokarir-intent-classifier")
   ```
6. **Integrasikan ke API Backend** — Wrap model dalam FastAPI endpoint:
   ```python
   @app.post("/classify")
   def classify_intent(text: str):
       result = classifier_dari_disk(text)[0]
       return {"intent": result["label"], "confidence": result["score"]}
   ```

---
*Dibuat untuk Capstone Project NeoKarir — Otak 1: Intent Classifier (IndoBERT)*